In [ ]:
# -*- coding: utf-8 -*-
"""PSYCHOMARKER - Conspiracy Detection (Colab Version with Drive Loading).ipynb

Automatically generated by Colab.
"""

"""
PSYCHOMARKER TASK - GOOGLE COLAB IMPLEMENTATION (IMPROVED)
Conspiracy Theory Detection using Unsloth Fine-tuning

IMPROVEMENTS FOR 80+ F1 SCORE:
- Few-shot learning with carefully selected examples
- Enhanced prompt engineering with reasoning steps
- Data augmentation and class balancing
- Improved inference with self-consistency
- Better hyperparameters

This script trains a model for Subtask 2: Binary classification (Yes/No)
Optimized for Google Colab with T4/V100 GPU
Files are loaded directly from Google Drive
"""

# ============================================================================
# GOOGLE COLAB SETUP - RUN THIS FIRST
# ============================================================================

# Install required packages
!pip install -q unsloth
!pip install -q xformers trl peft accelerate bitsandbytes
!pip install -q datasets scikit-learn tqdm pandas numpy

# ============================================================================
# IMPORTS
# ============================================================================

import os
import json
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from tqdm import tqdm
import random
from typing import List, Dict
import zipfile
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

# Unsloth imports
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

# Google Colab imports
from google.colab import drive
import shutil

print("\n" + "="*70)
print("PSYCHOMARKER - SUBTASK 2: CONSPIRACY DETECTION (IMPROVED)")
print("="*70)

# ============================================================================
# MOUNT GOOGLE DRIVE AND SET PATHS
# ============================================================================

print("\n" + "="*70)
print("MOUNTING GOOGLE DRIVE")
print("="*70)

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

# Set your Google Drive folder path
DRIVE_FOLDER = "/content/drive/MyDrive/psychomarker"

# Verify folder exists
if os.path.exists(DRIVE_FOLDER):
    print(f"✅ Found folder: {DRIVE_FOLDER}")

    # List all files in the folder
    print("\nFiles in your psychomarker folder:")
    files = os.listdir(DRIVE_FOLDER)
    for f in files:
        file_size = os.path.getsize(os.path.join(DRIVE_FOLDER, f)) / 1024  # KB
        print(f"   📄 {f} ({file_size:.1f} KB)")
else:
    print(f"❌ Folder not found: {DRIVE_FOLDER}")
    print("Please check the path or create the folder in your Google Drive")

# ============================================================================
# CONFIGURATION - CORRECTED FILE PATHS
# ============================================================================

# File paths - corrected for your actual files
TRAIN_FILE = os.path.join(DRIVE_FOLDER, "train_rehydrated_v2.jsonl")
DEV_FILE = os.path.join(DRIVE_FOLDER, "dev_rehydrated_v2.jsonl")
DEV_LABELS_FILE = os.path.join(DRIVE_FOLDER, "dev_public.jsonl")  # Using .jsonl extension
TEST_FILE = os.path.join(DRIVE_FOLDER, "test_rehydrated_v2.jsonl")

# Model settings
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

# IMPROVED Training settings
BATCH_SIZE = 2  # Will be auto-adjusted based on GPU
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1.5e-4
NUM_EPOCHS = 4
WARMUP_STEPS = 20
WEIGHT_DECAY = 0.01

# Data augmentation settings
USE_AUGMENTATION = True
BALANCE_CLASSES = True
OVERSAMPLE_RATIO = 0.7

print(f"\nModel: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

# ============================================================================
# CHECK GPU AND ADJUST SETTINGS
# ============================================================================

print("\n" + "="*70)
print("GPU INFORMATION")
print("="*70)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Adjust batch size based on GPU memory
    if "T4" in gpu_name:
        print("\n⚠️ T4 GPU detected - using optimized settings")
        BATCH_SIZE = 2
    elif "V100" in gpu_name:
        print("\n✅ V100 GPU detected - can use larger batch size")
        BATCH_SIZE = 4
    elif "A100" in gpu_name:
        print("\n🚀 A100 GPU detected - excellent!")
        BATCH_SIZE = 8
    else:
        print("\n⚠️ Unknown GPU - using conservative settings")
        BATCH_SIZE = 2
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    BATCH_SIZE = 1

print(f"\nAdjusted batch size: {BATCH_SIZE}")

# ============================================================================
# LOAD MODEL
# ============================================================================

print("\n" + "="*70)
print("LOADING MODEL")
print("="*70)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("✅ Base model loaded")

# ============================================================================
# CONFIGURE LORA
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)

print("✅ LoRA adapters configured")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.2f}%)")

# ============================================================================
# LOAD DATA FROM DRIVE
# ============================================================================

print("\n" + "="*70)
print("LOADING DATA FROM GOOGLE DRIVE")
print("="*70)

def load_jsonl(filepath):
    """Load JSONL file (one JSON object per line)"""
    data = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"⚠️ Error parsing line {line_num} in {os.path.basename(filepath)}: {e}")
                    print(f"   Line content: {line[:100]}...")
                    continue
        print(f"✅ Loaded {len(data)} examples from {os.path.basename(filepath)}")
    except FileNotFoundError:
        print(f"⚠️ File not found: {filepath}")
    except Exception as e:
        print(f"⚠️ Error loading {filepath}: {e}")
    return data

def load_labels_from_file(filepath):
    """Load labels from file - handles both JSON and JSONL formats"""
    labels = {}

    # First, check if file exists
    if not os.path.exists(filepath):
        print(f"⚠️ File not found: {filepath}")
        return labels

    # Read the file content
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    # Try parsing as JSON array first
    try:
        data = json.loads(content)
        if isinstance(data, list):
            # It's a JSON array
            for item in data:
                if '_id' in item and 'conspiracy' in item:
                    labels[item['_id']] = item['conspiracy']
            print(f"✅ Loaded {len(labels)} labels from JSON array in {os.path.basename(filepath)}")
            return labels
    except json.JSONDecodeError:
        pass

    # If not JSON array, try as JSONL (one JSON per line)
    try:
        # Reset file pointer
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    if '_id' in item and 'conspiracy' in item:
                        labels[item['_id']] = item['conspiracy']
                except json.JSONDecodeError as e:
                    print(f"⚠️ Error parsing line {line_num}: {e}")
                    continue
        print(f"✅ Loaded {len(labels)} labels from JSONL in {os.path.basename(filepath)}")
    except Exception as e:
        print(f"⚠️ Error processing file: {e}")

    return labels

# Load training data
train_data = load_jsonl(TRAIN_FILE)

# Load dev data for few-shot examples and evaluation
dev_data = load_jsonl(DEV_FILE)

# Load dev labels from file
print(f"\nAttempting to load labels from: {DEV_LABELS_FILE}")
dev_labels = load_labels_from_file(DEV_LABELS_FILE)

# Check if we have data to proceed
if len(train_data) == 0:
    print("\n❌ ERROR: Could not load training data!")
    raise SystemExit("Cannot proceed without training data")

if len(dev_data) == 0:
    print("\n❌ ERROR: Could not load dev data!")
    raise SystemExit("Cannot proceed without dev data")

if len(dev_labels) == 0:
    print(f"\n❌ ERROR: Could not load labels from {DEV_LABELS_FILE}")
    print("\nLet's debug the labels file:")
    try:
        with open(DEV_LABELS_FILE, 'r', encoding='utf-8') as f:
            content = f.read()
            print(f"File size: {len(content)} bytes")
            print(f"\nFirst 500 characters of the file:")
            print("-" * 50)
            print(content[:500])
            print("-" * 50)

            # Try to parse the entire file
            try:
                data = json.loads(content)
                print(f"\nSuccessfully parsed as JSON. Type: {type(data)}")
                if isinstance(data, list):
                    print(f"List length: {len(data)}")
                    if len(data) > 0:
                        print(f"First item keys: {data[0].keys()}")
            except json.JSONDecodeError as e:
                print(f"\nNot a valid JSON file. Error: {e}")
    except Exception as e:
        print(f"Error reading file: {e}")

    raise SystemExit("Cannot proceed without dev labels")

# Show sample of loaded labels
print(f"\nSample of loaded labels (first 5):")
for i, (doc_id, label) in enumerate(list(dev_labels.items())[:5]):
    print(f"   {doc_id}: {label}")

# Check label distribution in training data
yes_count = sum(1 for item in train_data if item.get('conspiracy') == 'Yes')
no_count = len(train_data) - yes_count
print(f"\nTraining data distribution:")
print(f"   Conspiracy (Yes): {yes_count} ({yes_count/len(train_data)*100:.1f}%)")
print(f"   Not conspiracy (No): {no_count} ({no_count/len(train_data)*100:.1f}%)")

# ============================================================================
# FEW-SHOT EXAMPLES (CAREFULLY SELECTED)
# ============================================================================

FEW_SHOT_EXAMPLES = """Here are some examples to guide your analysis:

Example 1:
Comment: "The government is using 5G towers to control our minds and spread diseases. Wake up sheeple!"
Analysis: This promotes conspiracy theories about 5G technology and government control without evidence.
Answer: Yes

Example 2:
Comment: "I disagree with the new policy. I think it will hurt small businesses based on economic research."
Analysis: This is a policy disagreement with reasoning, not a conspiracy theory.
Answer: No

Example 3:
Comment: "Big Pharma is hiding the cure for cancer to keep making money from treatments. They don't want us to know the truth!"
Analysis: This alleges a secret plot by pharmaceutical companies without credible evidence - a classic conspiracy theory.
Answer: Yes

Example 4:
Comment: "The moon landing was definitely faked in a Hollywood studio. The flag was waving and there's no wind on the moon!"
Analysis: This promotes the debunked moon landing conspiracy theory.
Answer: Yes

Example 5:
Comment: "I'm concerned about the side effects of this vaccine based on the clinical trial data published in the journal."
Analysis: This expresses concern based on scientific literature, not conspiracy thinking.
Answer: No

Example 6:
Comment: "Climate change is just a hoax invented by scientists to get more funding. It's all part of the global elite's agenda."
Analysis: This denies scientific consensus and alleges a coordinated conspiracy without evidence.
Answer: Yes

Example 7:
Comment: "I think the investigation was biased and we need better oversight. There are legitimate concerns about the process."
Analysis: This is criticism of a process with call for reform, not a conspiracy theory.
Answer: No

"""

# ============================================================================
# IMPROVED PROMPT WITH REASONING
# ============================================================================

CONSPIRACY_PROMPT_WITH_REASONING = """You are an expert at detecting conspiracy theories in social media content.

A conspiracy theory typically involves:
- Allegations of secret plots by powerful groups
- Claims of hidden "truths" being suppressed
- Lack of credible evidence
- Rejection of mainstream expert consensus
- Attribution of events to intentional coordination

{few_shot_examples}

Now analyze this new comment:

Comment: {text}

Think through these questions:
1. Does it allege a secret plot or coordination?
2. Does it claim hidden truths are being suppressed?
3. Does it reject expert consensus without credible evidence?
4. Does it attribute events to intentional malicious coordination?

Based on this analysis, is this comment promoting conspiracy theories?
Answer (Yes/No): """

# ============================================================================
# DATA AUGMENTATION
# ============================================================================

def augment_text(text, label):
    """Simple data augmentation for conspiracy texts"""
    augmented = []

    # Original
    augmented.append((text, label))

    # Only augment minority class (Yes)
    if label == 'Yes' and USE_AUGMENTATION:
        # Add slight variations (paraphrasing patterns)
        variations = []

        # Add contextual prefix
        if len(text.split()) > 10:
            variations.append(f"Someone said: {text}")
            variations.append(f"I read this: {text}")

        for var in variations:
            augmented.append((var, label))

    return augmented

# ============================================================================
# CLASS BALANCING
# ============================================================================

def balance_dataset(data, target_ratio=0.7):
    """Balance classes by oversampling minority class"""
    yes_samples = [x for x in data if x.get('conspiracy') == 'Yes']
    no_samples = [x for x in data if x.get('conspiracy') == 'No']

    print(f"\nClass balancing:")
    print(f"   Original - Yes: {len(yes_samples)}, No: {len(no_samples)}")

    if len(yes_samples) < len(no_samples):
        # Calculate how many Yes samples we need
        target_yes = int(len(no_samples) * target_ratio)
        oversample_count = target_yes - len(yes_samples)

        if oversample_count > 0:
            # Randomly oversample
            oversampled = random.choices(yes_samples, k=oversample_count)
            balanced_data = yes_samples + oversampled + no_samples
            random.shuffle(balanced_data)

            print(f"   Balanced - Yes: {target_yes}, No: {len(no_samples)}")
            print(f"   Added {oversample_count} oversampled Yes examples")

            return balanced_data

    return data

# ============================================================================
# PREPARE TRAINING DATA
# ============================================================================

print("\n" + "="*70)
print("PREPARING TRAINING DATA")
print("="*70)

# Balance classes if enabled
if BALANCE_CLASSES:
    train_data = balance_dataset(train_data, OVERSAMPLE_RATIO)
    print(f"✅ Balanced dataset: {len(train_data)} examples")
else:
    print(f"Using original dataset: {len(train_data)} examples")

def format_training_example(item, use_few_shot=True):
    """Format a single training example with optional few-shot"""
    text = item.get('text', '').strip()
    label = item.get('conspiracy', 'No')

    if not text:
        return None

    # Use improved prompt with reasoning
    if use_few_shot:
        prompt = CONSPIRACY_PROMPT_WITH_REASONING.format(
            few_shot_examples=FEW_SHOT_EXAMPLES,
            text=text
        )
    else:
        # Simpler prompt without few-shot for some variety
        prompt = CONSPIRACY_PROMPT_WITH_REASONING.format(
            few_shot_examples="",
            text=text
        )

    full_text = prompt + label

    return full_text

# Format training examples with augmentation
print("\nFormatting training examples...")
train_texts = []

for item in tqdm(train_data, desc="Formatting"):
    # Apply augmentation
    augmented_items = augment_text(item.get('text', ''), item.get('conspiracy', 'No'))

    for aug_text, aug_label in augmented_items:
        aug_item = {'text': aug_text, 'conspiracy': aug_label}

        # Mix few-shot and non-few-shot (80% few-shot, 20% without)
        use_few_shot = random.random() < 0.8
        formatted = format_training_example(aug_item, use_few_shot=use_few_shot)

        if formatted:
            train_texts.append(formatted)

print(f"✅ Formatted {len(train_texts)} training examples (with augmentation)")

# Create dataset
train_df = pd.DataFrame({'text': train_texts})
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)

# ============================================================================
# SHOW SAMPLE
# ============================================================================

print("\n" + "="*70)
print("SAMPLE TRAINING EXAMPLE")
print("="*70)
if train_texts:
    sample = train_texts[0]
    print(sample[:800] + ("..." if len(sample) > 800 else ""))
print("="*70)

# ============================================================================
# DATA COLLATOR
# ============================================================================

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ============================================================================
# IMPROVED TRAINING ARGUMENTS
# ============================================================================

# Save to both Drive and local
OUTPUT_DRIVE_PATH = os.path.join(DRIVE_FOLDER, "psychomarker_conspiracy_model")
OUTPUT_LOCAL_PATH = "/content/psychomarker_conspiracy_model"

training_args = SFTConfig(
    output_dir=OUTPUT_LOCAL_PATH,  # Save locally during training
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
    save_strategy="epoch",
    save_total_limit=2,
    group_by_length=True,
    torch_compile=False,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
)

print("\n" + "="*70)
print("TRAINING CONFIGURATION")
print("="*70)
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"LoRA rank: 32")
print(f"Data augmentation: {USE_AUGMENTATION}")
print(f"Class balancing: {BALANCE_CLASSES}")

# ============================================================================
# CREATE TRAINER
# ============================================================================

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
    data_collator=collator,
    dataset_text_field="text",
)

print("\n✅ Trainer initialized")

# ============================================================================
# TRAIN
# ============================================================================

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print("🎯 Goal: Detect conspiracy theories (Target F1: 80+)")
print("⏱️  Estimated time: 2-4 hours on Colab T4")
print("💾 Output will be saved to Google Drive")
print("="*70 + "\n")

trainer_stats = trainer.train()

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Final loss: {trainer_stats.training_loss:.4f}")

# ============================================================================
# SAVE MODEL TO DRIVE AND LOCAL
# ============================================================================

print("\n" + "="*70)
print("SAVING MODEL")
print("="*70)

# Save locally
model.save_pretrained(OUTPUT_LOCAL_PATH)
tokenizer.save_pretrained(OUTPUT_LOCAL_PATH)
print(f"✅ Model saved locally: {OUTPUT_LOCAL_PATH}")

# Save to Google Drive
model.save_pretrained(OUTPUT_DRIVE_PATH)
tokenizer.save_pretrained(OUTPUT_DRIVE_PATH)
print(f"✅ Model saved to Drive: {OUTPUT_DRIVE_PATH}")

# Create zip for easy download
!zip -r /content/psychomarker_model.zip /content/psychomarker_conspiracy_model
print("✅ Model zipped: /content/psychomarker_model.zip")

# ============================================================================
# INFERENCE WITH SELF-CONSISTENCY ON DEV SET
# ============================================================================

print("\n" + "="*70)
print("GENERATING DEV SET PREDICTIONS FOR EVALUATION")
print("="*70)

FastLanguageModel.for_inference(model)

def predict_with_confidence(model, tokenizer, text, device, num_samples=3):
    """Use self-consistency for more robust predictions"""

    prompt = CONSPIRACY_PROMPT_WITH_REASONING.format(
        few_shot_examples=FEW_SHOT_EXAMPLES,
        text=text
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    ).to(device)

    predictions = []

    with torch.no_grad():
        for _ in range(num_samples):
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                top_p=0.9,
            )

            full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer_part = full_response.split("Answer (Yes/No):")[-1].strip()

            if answer_part.lower().startswith('yes'):
                predictions.append('Yes')
            elif answer_part.lower().startswith('no'):
                predictions.append('No')
            else:
                predictions.append('No')

    vote_counts = Counter(predictions)
    return vote_counts.most_common(1)[0][0]

# Generate predictions
predictions = []
device = model.device

print("\n🔄 Processing dev set with self-consistency (3 samples per prediction)...")

for i, item in enumerate(tqdm(dev_data, desc="Predicting")):
    text = item.get('text', '').strip()

    if not text:
        conspiracy = 'No'
    else:
        conspiracy = predict_with_confidence(
            model, tokenizer, text, device, num_samples=3
        )

    predictions.append({
        "_id": item['_id'],
        "conspiracy": conspiracy,
        "markers": []
    })

print(f"\n✅ Generated {len(predictions)} predictions on dev set")

# ============================================================================
# EVALUATION AGAINST DEV LABELS
# ============================================================================

print("\n" + "="*70)
print("EVALUATING AGAINST DEV LABELS")
print("="*70)

pred_dict = {item['_id']: item['conspiracy'] for item in predictions}

y_true = []
y_pred = []
matched_ids = []

for item in dev_data:
    doc_id = item['_id']
    if doc_id in dev_labels:
        true_label = dev_labels[doc_id]
        if doc_id in pred_dict:
            pred_label = pred_dict[doc_id]
            y_true.append(true_label)
            y_pred.append(pred_label)
            matched_ids.append(doc_id)

print(f"Evaluated on {len(y_true)} matching documents")

if len(y_true) > 0:
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label='Yes')
    precision = precision_score(y_true, y_pred, pos_label='Yes')
    recall = recall_score(y_true, y_pred, pos_label='Yes')

    cm = confusion_matrix(y_true, y_pred, labels=['Yes', 'No'])

    print("\n" + "="*70)
    print("EVALUATION RESULTS")
    print("="*70)
    print(f"\n📊 Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"🎯 F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
    print(f"📈 Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"📉 Recall:    {recall:.4f} ({recall*100:.2f}%)")

    print("\n📊 Confusion Matrix:")
    print("              Predicted")
    print("              Yes    No")
    print(f"Actual Yes    {cm[0,0]:5d} {cm[0,1]:5d}")
    print(f"       No     {cm[1,0]:5d} {cm[1,1]:5d}")

    print("\n📋 Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Yes', 'No']))

    if f1 >= 0.80:
        print("\n✅ TARGET ACHIEVED: F1 Score >= 80%!")
    else:
        print(f"\n⚠️ Target not reached. Current F1: {f1*100:.1f}% (Target: 80%)")
        print(f"   Gap: {(80 - f1*100):.1f}%")

# ============================================================================
# STATISTICS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

yes_count = sum(1 for p in predictions if p['conspiracy'] == 'Yes')
no_count = len(predictions) - yes_count

print(f"Total dev set: {len(predictions)}")
print(f"  Conspiracy (Yes): {yes_count} ({yes_count/len(predictions)*100:.1f}%)")
print(f"  Not conspiracy (No): {no_count} ({no_count/len(predictions)*100:.1f}%)")

# ============================================================================
# SAVE DEV PREDICTIONS TO DRIVE AND LOCAL
# ============================================================================

print("\n" + "="*70)
print("SAVING DEV PREDICTIONS")
print("="*70)

DEV_OUTPUT_DRIVE = os.path.join(DRIVE_FOLDER, "dev_predictions.jsonl")
DEV_OUTPUT_LOCAL = "/content/dev_predictions.jsonl"

# Save locally
with open(DEV_OUTPUT_LOCAL, 'w', encoding='utf-8') as f:
    for pred in predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + '\n')

# Copy to Drive
shutil.copy(DEV_OUTPUT_LOCAL, DEV_OUTPUT_DRIVE)

print(f"✅ Dev predictions saved locally: {DEV_OUTPUT_LOCAL}")
print(f"✅ Dev predictions saved to Drive: {DEV_OUTPUT_DRIVE}")

# ============================================================================
# GENERATE TEST SET PREDICTIONS (OPTIONAL)
# ============================================================================

print("\n" + "="*70)
print("TEST SET PREDICTIONS")
print("="*70)

# Check if test file exists
if os.path.exists(TEST_FILE):
    print("Test file found. Generating test predictions...")

    test_data = load_jsonl(TEST_FILE)
    print(f"✅ Loaded {len(test_data)} test examples")

    # Generate test predictions
    test_predictions = []
    for i, item in enumerate(tqdm(test_data, desc="Predicting on test")):
        text = item.get('text', '').strip()

        if not text:
            conspiracy = 'No'
        else:
            conspiracy = predict_with_confidence(
                model, tokenizer, text, device, num_samples=3
            )

        test_predictions.append({
            "_id": item['_id'],
            "conspiracy": conspiracy,
            "markers": []
        })

    # Save test predictions
    TEST_OUTPUT_DRIVE = os.path.join(DRIVE_FOLDER, "submission.jsonl")
    TEST_OUTPUT_LOCAL = "/content/submission.jsonl"
    TEST_OUTPUT_ZIP = "/content/submission_conspiracy.zip"

    with open(TEST_OUTPUT_LOCAL, 'w', encoding='utf-8') as f:
        for pred in test_predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + '\n')

    shutil.copy(TEST_OUTPUT_LOCAL, TEST_OUTPUT_DRIVE)

    with zipfile.ZipFile(TEST_OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(TEST_OUTPUT_LOCAL, arcname="submission.jsonl")

    print(f"✅ Test submission saved locally: {TEST_OUTPUT_LOCAL}")
    print(f"✅ Test submission saved to Drive: {TEST_OUTPUT_DRIVE}")
    print(f"✅ Test ZIP created: {TEST_OUTPUT_ZIP}")
else:
    print("ℹ️ Test file not found. Skipping test predictions.")
    print(f"   (Looked for: {TEST_FILE})")

# ============================================================================
# FINAL OUTPUT
# ============================================================================

print("\n" + "="*70)
print("✅ PROCESS COMPLETE!")
print("="*70)
print(f"\n📊 Dev predictions file (Drive): {DEV_OUTPUT_DRIVE}")
print(f"📊 Dev predictions file (Local): {DEV_OUTPUT_LOCAL}")
print(f"📊 Documents evaluated: {len(y_true)}")
print(f"📁 Model saved (Drive): {OUTPUT_DRIVE_PATH}")
print(f"📁 Model saved (Local): {OUTPUT_LOCAL_PATH}")
print(f"📦 Model ZIP: /content/psychomarker_model.zip")

print("\n🚀 IMPROVEMENTS APPLIED:")
print("   ✓ Few-shot learning with 7 examples")
print("   ✓ Enhanced reasoning prompt")
print("   ✓ Class balancing (70% target ratio)")
print("   ✓ Data augmentation")
print("   ✓ Increased LoRA rank (32)")
print("   ✓ 4 epochs training")
print("   ✓ Self-consistency inference (3 samples)")
print("   ✓ Evaluation on dev set with metrics")

print("\n📤 FILES SAVED TO YOUR GOOGLE DRIVE:")
print(f"   {DRIVE_FOLDER}/")
print("   ├── psychomarker_conspiracy_model/  (trained model)")
print("   ├── dev_predictions.jsonl            (dev set results)")
if os.path.exists(TEST_FILE):
    print("   └── submission.jsonl                (test submission)")

print("\n📥 TO DOWNLOAD FILES:")
print("   1. In Colab, click the 'Files' tab on the left sidebar")
print("   2. Navigate to:")
print("      - /content/psychomarker_model.zip")
print("      - /content/dev_predictions.jsonl")
if os.path.exists(TEST_FILE):
    print("      - /content/submission_conspiracy.zip")
print("   3. Right-click and download")

print("\n" + "="*70)

# Optional: Auto-download
from google.colab import files

print("\nDo you want to download files now?")
download_now = input("Enter 'yes' to download files, or press Enter to skip: ").strip().lower()

if download_now == 'yes':
    files.download("/content/psychomarker_model.zip")
    files.download("/content/dev_predictions.jsonl")
    if os.path.exists(TEST_FILE):
        files.download("/content/submission_conspiracy.zip")